In [ ]:
# Cell 1
import os, sys
from pathlib import Path
os.chdir(Path(r"C:\Users\amand\OneDrive - Fisk University\Documents\ResumeBiasResearch"))
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from src.fairness.metrics import compute_all

sns.set_style("whitegrid")
data = np.load('data/processed/prepared_data.npz')
X_test = data['X_test']
y_test_blind = data['y_test_blind']

# Names come from the raw file
raw = np.load('data/raw/FairCVdb.npy', allow_pickle=True).item()
names_test = raw['Names Test']
bios_test = raw['Bios Test']  # shape (4800, 2) — [:, 0] has name, [:, 1] anonymized

In [ ]:
# Cell 2 — since profiles don't change between the two bios,
# the counterfactual experiment must be done on the BIO side.
# For FairCVdb, the profile features (51-dim vector) are identical across bios.
# So the "score change" is measured by changing the input representation.
#
# Approach: build two feature sets
#   A) X_test as-is (baseline)
#   B) X_test with name-derived features removed/perturbed
#
# For a text-based model, we'd need TF-IDF on the bios.
# Let's do BOTH: (1) profile-only model, and (2) bio-based model.

from sklearn.feature_extraction.text import TfidfVectorizer

# Extract bio variants
bios_with_name = [b[0] if isinstance(b, (list, np.ndarray)) else b for b in bios_test]
bios_anon = [b[1] if isinstance(b, (list, np.ndarray)) else b for b in bios_test]

print("Example pair:")
print("  WITH NAME:", bios_with_name[0][:200])
print("  ANON:     ", bios_anon[0][:200])

In [ ]:
# Cell 3 — Train a bio-only model to isolate the name effect
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
X_bio_with_name = vectorizer.fit_transform(bios_with_name)
X_bio_anon = vectorizer.transform(bios_anon)

clf = LogisticRegression(max_iter=2000, class_weight='balanced')
clf.fit(X_bio_with_name, y_test_blind)  # train on same set for demo — in real run, use train split

proba_with = clf.predict_proba(X_bio_with_name)[:, 1]
proba_anon = clf.predict_proba(X_bio_anon)[:, 1]

score_diff = proba_with - proba_anon
print(f"\nCounterfactual score differences (with-name minus anonymized):")
print(f"  Mean:   {score_diff.mean():+.4f}")
print(f"  Median: {np.median(score_diff):+.4f}")
print(f"  Std:    {score_diff.std():.4f}")
print(f"  |diff| > 0.05: {(np.abs(score_diff) > 0.05).mean()*100:.1f}% of candidates")

In [ ]:
# Cell 4 — Apply the same experiment to all three profile models
models = {
    'logistic_regression': joblib.load('results/models/logistic_regression_blind.pkl'),
    'random_forest':       joblib.load('results/models/random_forest_blind.pkl'),
    'xgboost':             joblib.load('results/models/xgboost_blind.pkl'),
}

# Since X_test is identical for both bio versions (profiles don't change),
# profile-only models produce the SAME output. This is a key finding:
# profile-only models are name-blind by construction.
#
# The bias enters through the LABELS, not the features.
# So we need a different comparison: train on biased labels vs blind labels
# and measure whether predictions diverge on the SAME inputs.

print("Profile-only models see the same features for both bio variants.")
print("→ Bias must come from the TRAINING LABELS, not the inputs.")
print("→ Next: train identical models on blind vs biased labels and compare.")

In [ ]:
# Cell 5 — The real experiment: same features, different label supervision
from src.models.train import get_models, evaluate_classifier

y_train_gender = data['y_train_gender']
y_train_ethnicity = data['y_train_ethnicity']
X_train = data['X_train']

results = []
for model_name, model_proto in get_models().items():
    for label_type, y_train, y_test in [
        ('blind', data['y_train_blind'], data['y_test_blind']),
        ('gender', y_train_gender, data['y_test_gender']),
        ('ethnicity', y_train_ethnicity, data['y_test_ethnicity']),
    ]:
        import copy
        m = copy.deepcopy(model_proto)
        m.fit(X_train, y_train)
        metrics = evaluate_classifier(m, X_test, y_test)
        results.append({
            'model': model_name,
            'label_type': label_type,
            **metrics,
        })

df = pd.DataFrame(results)
print(df.to_string(index=False))
df.to_csv('results/tables/model_by_label_type.csv', index=False)

In [ ]:
# Cell 6 — Visualize how model performance varies by label type
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pivot = df.pivot(index='model', columns='label_type', values='test_f1')
pivot = pivot[['blind', 'gender', 'ethnicity']]  # fixed order
pivot.plot(kind='bar', ax=axes[0], color=['steelblue', 'indianred', 'goldenrod'])
axes[0].set_title('F1 score by model and label type')
axes[0].set_ylabel('F1')
axes[0].legend(title='Labels trained on')

pivot_acc = df.pivot(index='model', columns='label_type', values='test_accuracy')[['blind','gender','ethnicity']]
pivot_acc.plot(kind='bar', ax=axes[1], color=['steelblue', 'indianred', 'goldenrod'])
axes[1].set_title('Accuracy by model and label type')
axes[1].set_ylabel('Accuracy')

pivot_auc = df.pivot(index='model', columns='label_type', values='test_auc')[['blind','gender','ethnicity']]
pivot_auc.plot(kind='bar', ax=axes[2], color=['steelblue', 'indianred', 'goldenrod'])
axes[2].set_title('AUC by model and label type')
axes[2].set_ylabel('AUC')

plt.tight_layout()
plt.savefig('results/figures/04_model_by_label_type.png', dpi=100, bbox_inches='tight')
plt.show()